# Solutions · 12 · Sampling and discretisation

Worked solutions to the exercises of [notebook 12](../notebooks/12_sampling_and_discretisation.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from englincontrol import systems as sy, discrete as dc, response as rs, linalg as la, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Discretise the prototype G at dt = 0.1, 0.5, 1.0 and 1.5 by ZOH and by Tustin and compare the step responses with the continuous one at the samples. Which method is exact at the samples whatever dt, and which preserves stability whatever dt?

In [2]:
G = sy.tf([4.0], [1.0, 1.2, 4.0])
for dt in (0.1, 0.5, 1.0, 1.5):
    n = int(round(6.0 / dt)) + 1
    t, y = rs.step(G, 6.0, n)
    err = {}
    for meth in ("zoh", "tustin"):
        Gd = dc.c2d(G, dt, meth)
        td, yd = rs.step(Gd, 6.0)
        err[meth] = (np.max(np.abs(yd[: len(y)] - y[: len(yd)])), bool(np.all(np.abs(Gd.poles()) < 1)))
    print(f"dt = {dt}: ZOH error at the samples {err['zoh'][0]:.1e} (stable {err['zoh'][1]}), Tustin error {err['tustin'][0]:.2f} (stable {err['tustin'][1]})")

dt = 0.1: ZOH error at the samples 4.4e-15 (stable True), Tustin error 0.06 (stable True)
dt = 0.5: ZOH error at the samples 2.2e-16 (stable True), Tustin error 0.26 (stable True)
dt = 1.0: ZOH error at the samples 1.1e-16 (stable True), Tustin error 0.38 (stable True)
dt = 1.5: ZOH error at the samples 2.2e-16 (stable True), Tustin error 0.54 (stable True)


The zero-order-hold model is exact at the samples at *every* Δt, because that is what the hold does to a
step; Tustin's model is only approximate, and its error grows with Δt - but it stays stable whatever Δt,
because the bilinear map sends the left half-plane into the unit disc. ZOH for plants, Tustin for controllers
remains the rule.

## Exercise 2

The ZOH discretisation of 1/s^3 at small dt has zeros outside the unit circle (sampling zeros). Find them with `dc.c2d(sy.tf([1.0], [1, 0, 0, 0]), dt).zeros()` for dt = 0.1 and explain why a dead-beat or inverse-based digital controller would fail here.

In [3]:
for dt in (0.1, 0.5, 1.0):
    Gd = dc.c2d(sy.tf([1.0], [1.0, 0.0, 0.0, 0.0]), dt)
    z = Gd.zeros()
    print(f"triple integrator at dt = {dt}: discrete zeros {np.round(z, 4)}, |z| = {np.round(np.abs(z), 4)}")

triple integrator at dt = 0.1: discrete zeros [-3.7321 -0.2679], |z| = [3.7321 0.2679]
triple integrator at dt = 0.5: discrete zeros [-3.7321 -0.2679], |z| = [3.7321 0.2679]
triple integrator at dt = 1.0: discrete zeros [-3.7321 -0.2679], |z| = [3.7321 0.2679]


Sampling 1/s³ with a hold creates two zeros at z = −2 ± √3 - one at −3.73, outside the unit circle, for every
Δt (Åström, Hagander and Sternby 1984): a discrete non-minimum-phase zero of a plant that has none in
continuous time. Any controller that cancels or inverts the plant (dead-beat, model-inverse feedforward)
would have an unstable pole at that zero; the sampling zeros are why such designs fail at fast sampling for
plants of relative degree three or more.

## Exercise 3

**Implement it yourself:** compute the ZOH matrices (Phi, Gamma) by the series expansion Phi = sum (A dt)^k/k!, Gamma = sum A^k dt^(k+1)/(k+1)! B truncated at 20 terms and compare with `linalg.expm_with_integral` for the DC motor at dt = 0.01 and at dt = 1.

In [4]:
mot = models.dc_motor()["system"]
A, B = mot.A, mot.B
for dt in (0.01, 1.0):
    Phi = np.eye(2); Gam = np.zeros((2, 1)); term = np.eye(2)
    for k in range(1, 21):
        Gam = Gam + term @ B * dt**k / float(np.prod(np.arange(1, k + 1)))
        term = term @ A * dt / k
        Phi = Phi + term
    Phi_ref, Gam_ref = la.expm_with_integral(A, B, dt)
    print(f"dt = {dt}: series (20 terms) vs library: Phi error {np.max(np.abs(Phi - Phi_ref)):.1e}, Gamma error {np.max(np.abs(Gam - Gam_ref)):.1e}  (|A dt| = {np.linalg.norm(A * dt, 1):.1f})")

dt = 0.01: series (20 terms) vs library: Phi error 1.7e-18, Gamma error 2.0e-04  (|A dt| = 0.1)
dt = 1.0: series (20 terms) vs library: Phi error 1.3e+01, Gamma error 3.0e-01  (|A dt| = 10.0)


For a small step the truncated series is exact to rounding; at Δt = 1 the matrix AΔt has norm 27 and twenty
terms are nowhere near enough - the series is still growing where it should be cancelling. The library's
scaling-and-squaring approach (notebook 18) removes the dependence on the step size.